# Accélérer une boucle Python avec Cython

Ce notebook part d'une fonction Python lente, le calcul d'une image de l'ensemble de Mandelbrot, et l'accélère pas à pas avec Cython :

1. Compiler le code Python tel quel
2. Typer les variables avec `cdef`
3. `cpdef` et une vue typée sur le tableau
4. Compiler un module avec `cythonize`
5. Pour aller plus loin : le mode Python pur

Chaque étape repart de la précédente et mesure son gain avec `%timeit`. Chaque exercice a une cellule pour votre code, puis des conseils supplémentaires et sa solution, repliés : ouvrez-les si besoin.

Cython traduit le code en C, que `gcc` compile : tous deux sont installés dans Colab comme dans le Codespace. La première cellule charge l'extension `cython` d'IPython, qui ajoute la commande magique `%%cython`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

%load_ext cython

## Le point de départ en Python

Pour chaque point `c = cx + i·cy` du plan, la suite `z ← z² + c`, partie de `z = 0`, reste bornée ou s'échappe vers l'infini. `iterations_py` compte les itérations avant que `|z|` dépasse 2, au plus `max_iter`. `mandelbrot_py` en fait une image : un pixel par point d'un rectangle du plan. L'ensemble de Mandelbrot est fait des points qui atteignent `max_iter`.

In [ ]:
# Les itérations de z = z² + c depuis z = 0, avec z = x + i·y et c = cx + i·cy,
# jusqu'à ce que |z| dépasse 2 (au plus max_iter).
def iterations_py(cx, cy, max_iter):
  x = y = 0.0
  for n in range(max_iter):
    if x * x + y * y > 4.0:
      return n
    x, y = x * x - y * y + cx, 2.0 * x * y + cy
  return max_iter


def mandelbrot_py(largeur, hauteur, max_iter):
  image = np.zeros((hauteur, largeur), dtype=np.int32)
  for i in range(hauteur):
    for j in range(largeur):
      # Le point du plan qu'occupe le pixel (i, j).
      cx = -2.0 + 3.0 * j / largeur
      cy = -1.2 + 2.4 * i / hauteur
      image[i, j] = iterations_py(cx, cy, max_iter)
  return image

In [ ]:
image = mandelbrot_py(480, 320, 100)
plt.imshow(image, cmap="magma")
plt.axis("off")
plt.show()

Le temps de référence, celui de Python :

In [ ]:
%timeit mandelbrot_py(480, 320, 100)

## 1. Compiler le code Python tel quel

Une cellule qui commence par `%%cython` est traduite en C par Cython, compilée, puis importée dans le notebook. Copiez-y `iterations_py` et `mandelbrot_py` sans changer leur code, en les renommant `iterations_v1` et `mandelbrot_v1`. Dans la seconde cellule, mesurez le temps de `mandelbrot_v1(480, 320, 100)` avec `%timeit`. Le gain est-il à la hauteur ?

In [ ]:
%%cython
# Votre code ici

In [ ]:
# Votre code ici

### Conseils supplémentaires

La cellule `%%cython` est un module à part : elle ne voit pas les variables du notebook, et importe donc NumPy elle-même. Renommez aussi l'appel à `iterations_py` dans `mandelbrot_v1`.

Un squelette à copier dans la première cellule de l'exercice, en remplaçant chaque `...` par le code de la version Python :

```cython
%%cython
import numpy as np


def iterations_v1(cx, cy, max_iter):
  ...


def mandelbrot_v1(largeur, hauteur, max_iter):
  ...
```

Puis, dans la seconde cellule :

```python
%timeit mandelbrot_v1(480, 320, 100)
```

### Solution

In [ ]:
%%cython
import numpy as np


def iterations_v1(cx, cy, max_iter):
  x = y = 0.0
  for n in range(max_iter):
    if x * x + y * y > 4.0:
      return n
    x, y = x * x - y * y + cx, 2.0 * x * y + cy
  return max_iter


def mandelbrot_v1(largeur, hauteur, max_iter):
  image = np.zeros((hauteur, largeur), dtype=np.int32)
  for i in range(hauteur):
    for j in range(largeur):
      cx = -2.0 + 3.0 * j / largeur
      cy = -1.2 + 2.4 * i / hauteur
      image[i, j] = iterations_v1(cx, cy, max_iter)
  return image

In [ ]:
%timeit mandelbrot_v1(480, 320, 100)
# Pas plus rapide que Python, souvent plus lent : sans types, le C produit
# manipule les mêmes objets Python que l'interpréteur.

## 2. Typer les variables avec `cdef`

`%%cython -a` compile la cellule et affiche le code annoté : plus une ligne est jaune, plus elle fait appel à l'interpréteur Python ; un clic sur le `+` d'une ligne montre le C produit. Écrivez `iterations_v2` et `mandelbrot_v2` dans une cellule `%%cython -a` :

- `iterations_v2` devient une fonction C, déclarée avec `cdef int`, aux arguments typés (`double cx`, `double cy`, `int max_iter`), comme ses variables locales (`x`, `y`, `n`) ;
- `mandelbrot_v2` reste une fonction `def`, qu'on appelle depuis Python, avec ses arguments et ses variables (`i`, `j`, `cx`, `cy`) typés.

Dans la seconde cellule, vérifiez avec `np.array_equal` que `mandelbrot_v2(480, 320, 100)` donne la même image que `image`, et mesurez son temps. Quelle ligne de la boucle reste jaune ?

In [ ]:
%%cython -a
# Votre code ici

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Les arguments se typent dans la signature, `def f(int n)` ; les variables locales avec `cdef`, en tête de fonction : `cdef double x = 0.0, y = 0.0`, `cdef int i, j`.
- Le type de retour d'une fonction `cdef` se place avant son nom : `cdef int f(...)`.
- Le reste du code ne change pas.

Un squelette à copier dans la première cellule de l'exercice, en remplaçant chaque `...` :

```cython
%%cython -a
import numpy as np


cdef int iterations_v2(double cx, double cy, int max_iter):
  cdef double x = 0.0, y = 0.0
  cdef int n
  ...


def mandelbrot_v2(int largeur, int hauteur, int max_iter):
  cdef int i, j
  cdef double cx, cy
  ...
```

Puis, dans la seconde cellule :

```python
print(np.array_equal(mandelbrot_v2(480, 320, 100), image))
%timeit mandelbrot_v2(480, 320, 100)
```

### Solution

In [ ]:
%%cython -a
import numpy as np


cdef int iterations_v2(double cx, double cy, int max_iter):
  cdef double x = 0.0, y = 0.0
  cdef int n
  for n in range(max_iter):
    if x * x + y * y > 4.0:
      return n
    x, y = x * x - y * y + cx, 2.0 * x * y + cy
  return max_iter


def mandelbrot_v2(int largeur, int hauteur, int max_iter):
  cdef int i, j
  cdef double cx, cy
  image = np.zeros((hauteur, largeur), dtype=np.int32)
  for i in range(hauteur):
    for j in range(largeur):
      cx = -2.0 + 3.0 * j / largeur
      cy = -1.2 + 2.4 * i / hauteur
      image[i, j] = iterations_v2(cx, cy, max_iter)
  return image

In [ ]:
print(np.array_equal(mandelbrot_v2(480, 320, 100), image))
%timeit mandelbrot_v2(480, 320, 100)
# De dix à vingt fois plus rapide que Python. Reste jaune la ligne
# image[i, j] = ... : l'indexation d'un tableau NumPy passe par Python.

## 3. `cpdef` et une vue typée sur le tableau

Une fonction `cdef` n'existe que côté C : le code Python ne voit pas `iterations_v2`.

In [ ]:
try:
  iterations_v2(1.0, 1.0, 100)
except NameError as erreur:
  print("Erreur :", erreur)

Écrivez `iterations_v3` et `mandelbrot_v3` dans une nouvelle cellule `%%cython -a` :

- déclarez `iterations_v3` avec `cpdef` : elle reste une fonction C rapide pour `mandelbrot_v3`, et devient aussi appelable depuis Python ;
- dans `mandelbrot_v3`, écrivez dans le tableau à travers une vue typée, `cdef int[:, :] vue = image`, plutôt que par `image[i, j]`.

Dans la seconde cellule, affichez `iterations_v3(0.0, 0.0, 100)` et `iterations_v3(1.0, 1.0, 100)`, qui valent `100` et `2`, vérifiez que l'image est la même, et mesurez le temps de `mandelbrot_v3`. Reste-t-il du jaune dans la boucle ?

**Documentation utile**

- https://cython.readthedocs.io/en/latest/src/userguide/memoryviews.html

In [ ]:
%%cython -a
# Votre code ici

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `cpdef int iterations_v3(...)` : seul le mot-clé change par rapport à `iterations_v2`.
- `int[:, :]` est une vue sur un tableau à deux dimensions de `int` C : le tableau `np.int32` que crée `np.zeros` en est un. `vue[i, j] = ...` devient alors un accès direct à la mémoire, et `image`, qui partage cette mémoire, contient le résultat.

Un squelette à copier dans la première cellule de l'exercice, en remplaçant chaque `...` :

```cython
%%cython -a
import numpy as np


cpdef int iterations_v3(double cx, double cy, int max_iter):
  ...


def mandelbrot_v3(int largeur, int hauteur, int max_iter):
  cdef int i, j
  cdef double cx, cy
  image = np.zeros((hauteur, largeur), dtype=np.int32)
  cdef int[:, :] vue = ...
  ...
  return image
```

Puis, dans la seconde cellule :

```python
print(iterations_v3(0.0, 0.0, 100), iterations_v3(1.0, 1.0, 100))
print(np.array_equal(mandelbrot_v3(480, 320, 100), image))
%timeit mandelbrot_v3(480, 320, 100)
```

### Solution

In [ ]:
%%cython -a
import numpy as np


cpdef int iterations_v3(double cx, double cy, int max_iter):
  cdef double x = 0.0, y = 0.0
  cdef int n
  for n in range(max_iter):
    if x * x + y * y > 4.0:
      return n
    x, y = x * x - y * y + cx, 2.0 * x * y + cy
  return max_iter


def mandelbrot_v3(int largeur, int hauteur, int max_iter):
  cdef int i, j
  cdef double cx, cy
  image = np.zeros((hauteur, largeur), dtype=np.int32)
  cdef int[:, :] vue = image
  for i in range(hauteur):
    for j in range(largeur):
      cx = -2.0 + 3.0 * j / largeur
      cy = -1.2 + 2.4 * i / hauteur
      vue[i, j] = iterations_v3(cx, cy, max_iter)
  return image

In [ ]:
print(iterations_v3(0.0, 0.0, 100), iterations_v3(1.0, 1.0, 100))
print(np.array_equal(mandelbrot_v3(480, 320, 100), image))
%timeit mandelbrot_v3(480, 320, 100)
# Encore deux à trois fois plus rapide : plus de jaune dans la boucle.

## 4. Compiler un module avec `cythonize`

Hors d'un notebook, le code Cython vit dans un fichier `.pyx`, que `cythonize -i` traduit en C et compile en un module importable, rangé à côté du fichier. Dans la première cellule, écrivez la version 3 dans `mandel.pyx` (la cellule commence déjà par `%%writefile mandel.pyx`), en nommant ses fonctions `iterations` et `mandelbrot`. Dans la seconde, compilez-le avec `!cythonize -i mandel.pyx`, importez le module `mandel` et mesurez le temps de `mandel.mandelbrot(480, 320, 100)`. `!cython -a mandel.pyx` écrit aussi l'annotation, dans `mandel.html`.

**Documentation utile**

- https://cython.readthedocs.io/en/latest/src/userguide/source_files_and_compilation.html#compiling-with-the-cythonize-command

In [ ]:
%%writefile mandel.pyx
# Votre code ici

In [ ]:
# Votre code ici

### Conseils supplémentaires

- La première ligne du fichier, `# cython: language_level=3`, dit à Cython que le code est du Python 3 : sans elle, Cython 3.0 prévient qu'il le suppose.
- `cythonize -i` (pour `--inplace`) crée `mandel.c`, puis le module compilé, `mandel.cpython-….so`, que `import mandel` charge.
- Python n'importe un module qu'une fois : après une modification de `mandel.pyx`, redémarrez la session avant de le compiler et de l'importer à nouveau.

Un squelette à copier dans les deux cellules de l'exercice, en remplaçant chaque `...` par le code de la version 3 :

```cython
%%writefile mandel.pyx
# cython: language_level=3
import numpy as np


cpdef int iterations(double cx, double cy, int max_iter):
  ...


def mandelbrot(int largeur, int hauteur, int max_iter):
  ...
```

Puis, dans la seconde cellule :

```python
!cythonize -i mandel.pyx
import mandel

print(np.array_equal(mandel.mandelbrot(480, 320, 100), image))
%timeit mandel.mandelbrot(480, 320, 100)
```

### Solution

In [ ]:
%%writefile mandel.pyx
# cython: language_level=3
import numpy as np


cpdef int iterations(double cx, double cy, int max_iter):
  cdef double x = 0.0, y = 0.0
  cdef int n
  for n in range(max_iter):
    if x * x + y * y > 4.0:
      return n
    x, y = x * x - y * y + cx, 2.0 * x * y + cy
  return max_iter


def mandelbrot(int largeur, int hauteur, int max_iter):
  cdef int i, j
  cdef double cx, cy
  image = np.zeros((hauteur, largeur), dtype=np.int32)
  cdef int[:, :] vue = image
  for i in range(hauteur):
    for j in range(largeur):
      cx = -2.0 + 3.0 * j / largeur
      cy = -1.2 + 2.4 * i / hauteur
      vue[i, j] = iterations(cx, cy, max_iter)
  return image

In [ ]:
!cythonize -i mandel.pyx
import mandel

print(np.array_equal(mandel.mandelbrot(480, 320, 100), image))
%timeit mandel.mandelbrot(480, 320, 100)

## 5. Pour aller plus loin : le mode Python pur

Depuis Cython 3, un fichier `.py` ordinaire peut porter ses types en annotations : `x: cython.double`, `@cython.cfunc` pour une fonction `cdef`, `@cython.ccall` pour une fonction `cpdef`, `cython.int[:, :]` pour une vue typée. Le fichier reste du Python valide, qui s'exécute sans compilation, et `cythonize -i` le compile comme un `.pyx`. Dans la première cellule, écrivez ainsi la version 3 dans `mandel_pur.py`. Dans la seconde, compilez-la, importez `mandel_pur`, affichez `mandel_pur.__file__` pour voir quel fichier Python a chargé, et mesurez son temps.

**Documentation utile**

- https://cython.readthedocs.io/en/latest/src/tutorial/pure.html

In [ ]:
%%writefile mandel_pur.py
# Votre code ici

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Chaque variable typée se déclare par une annotation, une par ligne : `i: cython.int`.
- Le type de retour d'une fonction est son annotation de retour : `-> cython.int`.
- Quand `mandel_pur.py` et le module compilé sont côte à côte, `import mandel_pur` charge le module compilé : `__file__` finit par `.so`.

Un squelette à copier dans les deux cellules de l'exercice, en remplaçant chaque `...` :

```python
%%writefile mandel_pur.py
# cython: language_level=3
import cython
import numpy as np


@cython.ccall
def iterations(
  cx: cython.double, cy: cython.double, max_iter: cython.int
) -> cython.int:
  x: cython.double = 0.0
  ...


def mandelbrot(largeur: cython.int, hauteur: cython.int, max_iter: cython.int):
  i: cython.int
  ...
  vue: cython.int[:, :] = image
  ...
```

Puis, dans la seconde cellule :

```python
!cythonize -i mandel_pur.py
import mandel_pur

print(mandel_pur.__file__)
print(np.array_equal(mandel_pur.mandelbrot(480, 320, 100), image))
%timeit mandel_pur.mandelbrot(480, 320, 100)
```

### Solution

In [ ]:
%%writefile mandel_pur.py
# cython: language_level=3
import cython
import numpy as np


@cython.ccall
def iterations(
  cx: cython.double, cy: cython.double, max_iter: cython.int
) -> cython.int:
  x: cython.double = 0.0
  y: cython.double = 0.0
  n: cython.int
  for n in range(max_iter):
    if x * x + y * y > 4.0:
      return n
    x, y = x * x - y * y + cx, 2.0 * x * y + cy
  return max_iter


def mandelbrot(largeur: cython.int, hauteur: cython.int, max_iter: cython.int):
  i: cython.int
  j: cython.int
  cx: cython.double
  cy: cython.double
  image = np.zeros((hauteur, largeur), dtype=np.int32)
  vue: cython.int[:, :] = image
  for i in range(hauteur):
    for j in range(largeur):
      cx = -2.0 + 3.0 * j / largeur
      cy = -1.2 + 2.4 * i / hauteur
      vue[i, j] = iterations(cx, cy, max_iter)
  return image

In [ ]:
!cythonize -i mandel_pur.py
import mandel_pur

print(mandel_pur.__file__)
print(np.array_equal(mandel_pur.mandelbrot(480, 320, 100), image))
%timeit mandel_pur.mandelbrot(480, 320, 100)
# Le module importé est le .so compilé, aussi rapide que mandel.